** Introduction**


This lab tutorial presents a comprehensive distributed time-series forecasting pipeline built with PySpark, aimed at predicting Bitcoin closing prices. The workflow includes:

Loading the dataset and examining its schema
Data preprocessing and chronological ordering
Temporal feature engineering using Window functions
Performing a time-aware train-test split to avoid data leakage
Training models such as Linear Regression and Gradient Boosted Trees
Producing predictions and evaluating performance using RMSE, MAE, and R²

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder \
    .appName('Bitcoin_Forecasting') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')

print('SparkSession initialised successfully.')
print(f'Spark version: {spark.version}')

SparkSession initialised successfully.
Spark version: 4.0.2


In [ ]:
import os

# Define the original and new file paths
original_path = '/content/btcusd_1-min_data[1].csv'
new_path = '/content/btcusd_1-min_data_1.csv' # Renamed to remove special characters

# Check if the original file exists and rename it
if os.path.exists(original_path):
    os.rename(original_path, new_path)
    print(f"Renamed '{original_path}' to '{new_path}'")
else:
    print(f"Original file '{original_path}' not found. Cannot rename. Attempting to use original path.")
    new_path = original_path # Fallback in case renaming fails or file is already renamed

btc_df = spark.read.csv(
    new_path,
    header=True,
    inferSchema=True
)

print(f'Total records loaded: {btc_df.count():,}')
print('\nSchema:')
btc_df.printSchema()
print('\nFirst 5 rows:')
btc_df.show(5, truncate=False)

Renamed '/content/btcusd_1-min_data[1].csv' to '/content/btcusd_1-min_data_1.csv'
Total records loaded: 7,522,245

Schema:
root
 |-- Timestamp: double (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: double (nullable = true)


First 5 rows:
+------------+----+----+----+-----+------+
|Timestamp   |Open|High|Low |Close|Volume|
+------------+----+----+----+-----+------+
|1.32541206E9|4.58|4.58|4.58|4.58 |0.0   |
|1.32541212E9|4.58|4.58|4.58|4.58 |0.0   |
|1.32541218E9|4.58|4.58|4.58|4.58 |0.0   |
|1.32541224E9|4.58|4.58|4.58|4.58 |0.0   |
|1.3254123E9 |4.58|4.58|4.58|4.58 |0.0   |
+------------+----+----+----+-----+------+
only showing top 5 rows


In [ ]:
# Drop rows where Close is null
btc_clean = btc_df.dropna(subset=['Close'])

# Convert Unix timestamp to datetime
btc_clean = btc_clean.withColumn(
    'datetime',
    F.to_timestamp(F.from_unixtime(F.col('Timestamp')))
).withColumn(
    'date',
    F.to_date(F.col('datetime'))
)

# Aggregate to daily OHLCV — take last Close per calendar day
daily_df = btc_clean.groupBy('date').agg(
    F.last('Open').alias('Open'),
    F.max('High').alias('High'),
    F.min('Low').alias('Low'),
    F.last('Close').alias('Close'),
    F.sum('Volume').alias('Volume')
).orderBy('date')

print(f'Daily records after preprocessing: {daily_df.count()}')
print('\nDescriptive statistics:')
daily_df.describe(['Close', 'Volume']).show()

Daily records after preprocessing: 5226

Descriptive statistics:
+-------+------------------+-----------------+
|summary|             Close|           Volume|
+-------+------------------+-----------------+
|  count|              5226|             5226|
|   mean|22832.987556448523|7262.793702638632|
| stddev| 31017.52865328429|8907.958995304449|
|    min|              4.38|              0.0|
|    max|          124728.0|  127286.48653306|
+-------+------------------+-----------------+



In [ ]:
w = Window.orderBy('date')

daily_feat = daily_df \
    .withColumn('lag_1',  F.lag('Close', 1).over(w)) \
    .withColumn('lag_3',  F.lag('Close', 3).over(w)) \
    .withColumn('lag_7',  F.lag('Close', 7).over(w)) \
    .withColumn('lag_14', F.lag('Close', 14).over(w)) \
    .withColumn('lag_30', F.lag('Close', 30).over(w)) \
    .withColumn('roll_mean_7',  F.avg('Close').over(w.rowsBetween(-6, 0))) \
    .withColumn('roll_mean_14', F.avg('Close').over(w.rowsBetween(-13, 0))) \
    .withColumn('roll_std_7',   F.stddev('Close').over(w.rowsBetween(-6, 0))) \
    .withColumn('roll_std_14',  F.stddev('Close').over(w.rowsBetween(-13, 0))) \
    .withColumn('daily_return', (F.col('Close') - F.col('lag_1')) / F.col('lag_1')) \
    .withColumn('return_7d',    (F.col('Close') - F.col('lag_7'))  / F.col('lag_7')) \
    .withColumn('label', F.lead('Close', 1).over(w))  # next-day Close as target

daily_feat = daily_feat.dropna()

print(f'Records after feature engineering: {daily_feat.count()}')
print('\nSample feature rows:')
daily_feat.select(
    'date', 'Close', 'lag_1', 'lag_7',
    'roll_mean_7', 'roll_std_7', 'daily_return', 'label'
).show(5)

Records after feature engineering: 5195

Sample feature rows:
+----------+-----+-----+-----+-----------------+-------------------+--------------------+-----+
|      date|Close|lag_1|lag_7|      roll_mean_7|         roll_std_7|        daily_return|label|
+----------+-----+-----+-----+-----------------+-------------------+--------------------+-----+
|2012-01-31| 5.55| 5.58| 6.55|5.727142857142857|0.43649033153531525|-0.00537634408602155| 5.99|
|2012-02-01| 5.99| 5.55|  6.0|5.725714285714285|0.43546362813508466| 0.07927927927927936| 6.26|
|2012-02-02| 6.26| 5.99| 6.27|5.724285714285713|  0.433391937429126| 0.04507512520868106| 6.29|
|2012-02-03| 6.29| 6.26| 5.88|5.782857142857142| 0.4828289650837131|0.004792332268370647|  6.5|
|2012-02-04|  6.5| 6.29| 4.91|             6.01|0.36285901761795425| 0.03338632750397456|  5.7|
+----------+-----+-----+-----+-----------------+-------------------+--------------------+-----+
only showing top 5 rows


In [ ]:
feature_cols = [
    'Open', 'High', 'Low', 'Volume',
    'lag_1', 'lag_3', 'lag_7', 'lag_14', 'lag_30',
    'roll_mean_7', 'roll_mean_14', 'roll_std_7', 'roll_std_14',
    'daily_return', 'return_7d'
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')
assembled = assembler.transform(daily_feat).select('date', 'features', 'label')

# Chronological 80/20 split — no random shuffling to prevent data leakage
total = assembled.count()
train_count = int(total * 0.80)

assembled_indexed = assembled.withColumn('row_id', F.monotonically_increasing_id())
train_df = assembled_indexed.filter(F.col('row_id') < train_count)
test_df  = assembled_indexed.filter(F.col('row_id') >= train_count)

print(f'Training samples : {train_df.count()}')
print(f'Test samples     : {test_df.count()}')
print(f'Train date range : {train_df.agg(F.min("date"), F.max("date")).collect()}')
print(f'Test date range  : {test_df.agg(F.min("date"), F.max("date")).collect()}')

In [ ]:
lr = LinearRegression(
    featuresCol='features',
    labelCol='label',
    maxIter=200,
    regParam=0.05,
    elasticNetParam=0.3
)

lr_model = lr.fit(train_df)

print('Linear Regression training complete.')
print(f'Training RMSE  : {lr_model.summary.rootMeanSquaredError:.4f}')
print(f'Training R2    : {lr_model.summary.r2:.4f}')
print(f'Coefficients   : {lr_model.coefficients}')

In [ ]:
gbt = GBTRegressor(
    featuresCol='features',
    labelCol='label',
    maxIter=100,
    maxDepth=5,
    stepSize=0.1,
    subsamplingRate=0.8
)

gbt_model = gbt.fit(train_df)

print('GBT training complete.')
print('Feature importances (top 5):')
fi = sorted(zip(feature_cols, gbt_model.featureImportances), key=lambda x: -x[1])
for feat, score in fi[:5]:
    print(f'  {feat:<18} {score:.4f}')

In [ ]:
lr_preds  = lr_model.transform(test_df).withColumnRenamed('prediction', 'lr_pred')
gbt_preds = gbt_model.transform(test_df).withColumnRenamed('prediction', 'gbt_pred')

# Join both sets of predictions for side-by-side comparison
comparison = lr_preds.select('date', 'label', 'lr_pred') \
    .join(gbt_preds.select('date', 'gbt_pred'), on='date', how='inner') \
    .orderBy('date')

comparison.select(
    'date',
    F.round('label', 2).alias('Actual_Close'),
    F.round('lr_pred', 2).alias('LR_Prediction'),
    F.round('gbt_pred', 2).alias('GBT_Prediction')
).show(10)

In [ ]:
evaluator_rmse = RegressionEvaluator(
    labelCol='label', predictionCol='prediction', metricName='rmse')
evaluator_mae  = RegressionEvaluator(
    labelCol='label', predictionCol='prediction', metricName='mae')
evaluator_r2   = RegressionEvaluator(
    labelCol='label', predictionCol='prediction', metricName='r2')

# Re-generate predictions with the original 'prediction' column name for the evaluators
lr_eval_preds  = lr_model.transform(test_df)
gbt_eval_preds = gbt_model.transform(test_df)

for name, preds in [('Linear Regression', lr_eval_preds), ('GBT', gbt_eval_preds)]:
    rmse = evaluator_rmse.evaluate(preds)
    mae  = evaluator_mae.evaluate(preds)
    r2   = evaluator_r2.evaluate(preds)
    print(f'--- {name} ---')
    print(f'  RMSE : {rmse:.4f}')
    print(f'  MAE  : {mae:.4f}')
    print(f'  R2   : {r2:.4f}')
    print()

In [ ]:
spark.stop()
print('SparkSession stopped.')